In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

presto_port = '80'
# username = 'airflow-user'
username = 'jatin.rajani1@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
connection4= presto.connect('processing.processing.data.production.internal',presto_port,username)


In [28]:
city_name = 'Jaipur'
start_date='2024-02-09'
end_date='2024-02-14'

In [29]:
def all_services(start_date,end_date,city_name,service_name):
    all_dist=f"""WITH service_mapping AS ( 
        SELECT 
            service_detail_id,
            service_level as service_name,
            service_category,
            service_id,
            city_display_name as city,
            city_id
        FROM 
            datasets.service_mapping
        WHERE 
            city_display_name='{city_name}'
            and service_level='{service_name}'
    ),
    
    fe_tbl as (
            SELECT
                fare_estimate_id fid,
                api_context,final_amount as final_amount_fare,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                epoch fe,ride_distance as d1, estimated_ride_time as t1,
                CASE WHEN SUBSTR(user_id, -1) IN ('0','2','4','6','8','a','c','e') THEN 'TEST' ELSE 'CONTROL' END group_tc
            FROM
                pricing.fare_estimates_enriched 
            WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
               
    ),
    orderlogs as 
    (SELECT *,u.id_array as fare_estimate_id
    FROM 
    ( SELECT order_id, (amount-prev_due) as order_amount,customer_id,captain_id,
             date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
             epoch,estimate_id,estimate_ids,ride_time as t3,
             fare_recalculated_reason, all_paths,
             spd_fraud_flag,order_status,
             service_obj_service_name as service_name,
             fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
             cast(json_parse(estimate_ids) as array<varchar>) AS id_array
      FROM orders.order_logs_snapshot
      WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)) as t
    CROSS JOIN UNNEST(t.id_array) AS u(id_array)),
    immutable AS (
        SELECT 
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            order_id,
            cast (event_time as double) event_time
        FROM 
            orders.order_logs_immutable 
       WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_type = 'started'
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
    ),
    
    nav_metrics AS (
        SELECT 
            event_props_order_id,
            cast  (event_props_epoch as double) event_props_epoch,
            try(cast(event_props_distance_in_meters as double) )/ 1000 d2
        FROM
            clevertap.captain_inappnavmetrics_immutable
        WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_props_order_status = 'started'
    ),
    
    distance2 AS (
        SELECT 
            order_date,
            order_id,
            d2
        FROM 
            ( 
            select 
                order_date,
                imu.order_id,
                date_diff('minute', from_unixtime(event_time/1000), from_unixtime(event_props_epoch/1000)) AS minDiff,
                "d2"
            from 
                immutable imu 
            join 
                nav_metrics nav 
                ON imu.order_id = nav.event_props_order_id
            )
            --and event_props_epoch >= event_time )
        WHERE abs(minDiff) <= 1
    ),
    done as  (select 
               distinct order_id,d1,t1,t3,epoch,fare_recalculated_reason,service_name,rn,api_context, (fe_amount-order_amount) as amt_diff, all_paths,fid,group_tc,spd_fraud_flag,order_status
           
               from 
              (SELECT *,row_number () over(partition by order_id order by fe) rn from 
              (SELECT  *
               from orderlogs join fe_tbl
               on fare_estimate_id = fid 
               and service_detail_id = service_id_fare)
               )
               where rn =1
     ),
    dthree as (select distinct order_id, CASE
            WHEN selectedroute = 'hFRoute' THEN hfdistance
            WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
            WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
            ELSE NULL -- Handle other cases if needed
        END AS d3, selectedroute
        from
    (select * from (
    select *, row_number () over(partition by order_id order by updatedAt) rn from (
     select yyyymmdd,orderid as order_id, cast(hfdistance as double)*1000 as hfdistance,cast(pingdensity as double)*100 as pingdensity,cast(directionsresponsedistance as double) as directionsresponsedistance,
                 cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,selectedroute,updatedAt
                 from raw.captain_distance_response
                 where yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                   
                ))
                where rn = 1
                 )),
                 
    merged_tbl as
    (
    
    
        select done.order_id,d2,order_date,d3,d1,group_tc,fare_recalculated_reason,spd_fraud_flag,order_status,service_name,
        CASE
            WHEN d2 IS NOT NULL then d1-d2
            END AS fe_vs_nav,
            d1-(d3*1.00/1000.00) as Est_Nav,
            case WHEN d2 IS NOT NULL then d2-(d3*1.00/1000.00)
            END AS nav_actual
        from done
        left join dthree
        on done.order_id=dthree.order_id
        left join distance2
        on done.order_id=distance2.order_id
    
    
    
    )
    
    
    select service_name,group_tc,count(distinct order_id) as gross_orders,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end) as net_orders,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end) as route_diff_orders,
     
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end) as fe_nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='b. Less_than_500m' then order_id end) as fe_nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='c. Greater_than_500m' then order_id end) as fe_nav_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end) as nav_actual_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='b. Less_than_500m' then order_id end) as nav_actual_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='c. Greater_than_500m' then order_id end) as nav_actual_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='a. Within_500_m' then order_id end) as Est_Nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='b. Less_than_500m' then order_id end) as Est_Nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='c. Greater_than_500m' then order_id end) as Est_Nav_bucket_grt_500m
    
    
    
    from(
            select *,case when d2 is not NULL and fe_vs_nav>=-0.5 and fe_vs_nav<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and fe_vs_nav<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and fe_vs_nav>0.5 then 'c. Greater_than_500m' end as fe_vs_nav_bucket,
            case when  d2 is not NULL and nav_actual>=-0.5 and nav_actual<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and nav_actual<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and nav_actual>0.5 then 'c. Greater_than_500m' end as nav_actual_bucket,
            case when Est_Nav>=-0.5 and Est_Nav<=0.5 then 'a. Within_500_m'
            when Est_Nav<-0.5 then 'b. Less_than_500m'
            when Est_Nav>0.5 then 'c. Greater_than_500m' end as Est_Nav_bucket
            from merged_tbl
            group by 1,2,3,4,5,6,7,8,9,10,11,12,13
    )
    group by 1,2
    """
    d1_df=pd.read_sql(all_dist,connection4)
    return d1_df

In [30]:
total_df=pd.DataFrame()
for i in ['Link','Auto','Bike Lite']:
    print(i)
    d1=all_services(start_date,end_date,city_name,i)
    total_df=pd.concat([total_df,d1])

Link
Auto
Bike Lite


In [ ]:
total_df

In [31]:
total_df.to_csv('total_df_all_v1.csv')

In [16]:
total_df.to_csv('total_df_all.csv')

In [ ]:
(case when cast(substr(hhmmss,1,2) as real) >= 8 and cast(substr(hhmmss,1,2) as real) <= 11 then 'Morning_Peak'
        when cast(substr(hhmmss,1,2) as real) > 11 and cast(substr(hhmmss,1,2) as real) < 17 then 'Afternoon'
        when cast(substr(hhmmss,1,2) as real) >= 17 and cast(substr(hhmmss,1,2) as real) <= 21 then 'Evening_Peak'
        when cast(substr(hhmmss,1,2) as real) >= 0 and cast(substr(hhmmss,1,2) as real) < 8 then 'Rest_Morning'
        else 'Rest_Evening' end) as time_bucket, map_riders_count,
        (case when distance_final_distance >= 0 and distance_final_distance < 5 then 'Short'
        when distance_final_distance >= 5 and distance_final_distance < 9 then 'Medium'
        when distance_final_distance >= 9 then 'Long' end) as lm_bucket,

In [33]:
def all_services_temporal_dist(start_date,end_date,city_name,service_name):
    all_dist=f"""WITH service_mapping AS ( 
        SELECT 
            service_detail_id,
            service_level as service_name,
            service_category,
            service_id,
            city_display_name as city,
            city_id
        FROM 
            datasets.service_mapping
        WHERE 
            city_display_name='{city_name}'
            and service_level='{service_name}'
    ),
    
    fe_tbl as (
            SELECT
                fare_estimate_id fid,
                api_context,final_amount as final_amount_fare,
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
                cast(final_amount AS double) AS fe_amount,service_detail_id as service_id_fare,
                epoch fe,ride_distance as d1, estimated_ride_time as t1,
                CASE WHEN SUBSTR(user_id, -1) IN ('0','2','4','6','8','a','c','e') THEN 'TEST' ELSE 'CONTROL' END group_tc
            FROM
                pricing.fare_estimates_enriched 
            WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
               
    ),
    orderlogs as 
    (SELECT *,u.id_array as fare_estimate_id
    FROM 
    ( SELECT order_id, (amount-prev_due) as order_amount,customer_id,captain_id,
             date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y-%m-%d') AS orderdate,
             epoch,estimate_id,estimate_ids,ride_time as t3,
             fare_recalculated_reason, all_paths,
             spd_fraud_flag,order_status,
             service_obj_service_name as service_name,
            (case when cast(substr(hhmmss,1,2) as real) >= 8 and cast(substr(hhmmss,1,2) as real) <= 11 then 'Morning_Peak'
            when cast(substr(hhmmss,1,2) as real) > 11 and cast(substr(hhmmss,1,2) as real) < 17 then 'Afternoon'
            when cast(substr(hhmmss,1,2) as real) >= 17 and cast(substr(hhmmss,1,2) as real) <= 21 then 'Evening_Peak'
            when cast(substr(hhmmss,1,2) as real) >= 0 and cast(substr(hhmmss,1,2) as real) < 8 then 'Rest_Morning'
            else 'Rest_Evening' end) as time_bucket,
            (case when distance_final_distance >= 0 and distance_final_distance < 5 then 'Short'
            when distance_final_distance >= 5 and distance_final_distance < 9 then 'Medium'
            when distance_final_distance >= 9 then 'Long' end) as lm_bucket,
             fare_recalculated_diff_amount AS fe_re_diff_amount,service_detail_id,
             cast(json_parse(estimate_ids) as array<varchar>) AS id_array
      FROM orders.order_logs_snapshot
      WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)) as t
    CROSS JOIN UNNEST(t.id_array) AS u(id_array)),
    immutable AS (
        SELECT 
            DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
            order_id,
            cast (event_time as double) event_time
        FROM 
            orders.order_logs_immutable 
       WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_type = 'started'
            AND city_name IN (SELECT city FROM service_mapping)
            AND service_obj_service_name IN (SELECT service_name FROM service_mapping)
    ),
    
    nav_metrics AS (
        SELECT 
            event_props_order_id,
            cast  (event_props_epoch as double) event_props_epoch,
            try(cast(event_props_distance_in_meters as double) )/ 1000 d2
        FROM
            clevertap.captain_inappnavmetrics_immutable
        WHERE 
            yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
            AND event_props_order_status = 'started'
    ),
    
    distance2 AS (
        SELECT 
            order_date,
            order_id,
            d2
        FROM 
            ( 
            select 
                order_date,
                imu.order_id,
                date_diff('minute', from_unixtime(event_time/1000), from_unixtime(event_props_epoch/1000)) AS minDiff,
                "d2"
            from 
                immutable imu 
            join 
                nav_metrics nav 
                ON imu.order_id = nav.event_props_order_id
            )
            --and event_props_epoch >= event_time )
        WHERE abs(minDiff) <= 1
    ),
    done as  (select 
               distinct order_id,d1,t1,t3,epoch,fare_recalculated_reason,service_name,rn,api_context, (fe_amount-order_amount) as amt_diff, all_paths,fid,group_tc,spd_fraud_flag,order_status,
               lm_bucket,time_bucket
           
               from 
              (SELECT *,row_number () over(partition by order_id order by fe) rn from 
              (SELECT  *
               from orderlogs join fe_tbl
               on fare_estimate_id = fid 
               and service_detail_id = service_id_fare)
               )
               where rn =1
     ),
    dthree as (select distinct order_id, CASE
            WHEN selectedroute = 'hFRoute' THEN hfdistance
            WHEN selectedroute = 'actualRoute' THEN directionsresponsedistance
            WHEN selectedroute = 'estimatedRoute' THEN final_estimated_directions_response_distance
            ELSE NULL -- Handle other cases if needed
        END AS d3, selectedroute
        from
    (select * from (
    select *, row_number () over(partition by order_id order by updatedAt) rn from (
     select yyyymmdd,orderid as order_id, cast(hfdistance as double)*1000 as hfdistance,cast(pingdensity as double)*100 as pingdensity,cast(directionsresponsedistance as double) as directionsresponsedistance,
                 cast(final_estimated_directions_response_distance as double) as final_estimated_directions_response_distance,selectedroute,updatedAt
                 from raw.captain_distance_response
                 where yyyymmdd BETWEEN DATE_FORMAT(date('{start_date}'),'%Y%m%d') 
            AND DATE_FORMAT(date('{end_date}'),'%Y%m%d') 
                   
                ))
                where rn = 1
                 )),
                 
    merged_tbl as
    (
    
    
        select done.order_id,d2,order_date,d3,d1,group_tc,fare_recalculated_reason,spd_fraud_flag,order_status,service_name,lm_bucket,
        time_bucket,
        CASE
            WHEN d2 IS NOT NULL then d1-d2
            END AS fe_vs_nav,
            d1-(d3*1.00/1000.00) as Est_Nav,
            case WHEN d2 IS NOT NULL then d2-(d3*1.00/1000.00)
            END AS nav_actual
        from done
        left join dthree
        on done.order_id=dthree.order_id
        left join distance2
        on done.order_id=distance2.order_id
    
    
    
    )
    
    
    select service_name,group_tc,lm_bucket,time_bucket,count(distinct order_id) as gross_orders,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True then order_id end) as net_orders,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fare_recalculated_reason='routeDiff' then order_id end) as route_diff_orders,
     
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='a. Within_500_m' then order_id end) as fe_nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='b. Less_than_500m' then order_id end) as fe_nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and fe_vs_nav_bucket='c. Greater_than_500m' then order_id end) as fe_nav_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='a. Within_500_m' then order_id end) as nav_actual_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='b. Less_than_500m' then order_id end) as nav_actual_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and nav_actual_bucket='c. Greater_than_500m' then order_id end) as nav_actual_bucket_grt_500m,
    
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='a. Within_500_m' then order_id end) as Est_Nav_bucket_within_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='b. Less_than_500m' then order_id end) as Est_Nav_bucket_less_500m,
    count(distinct case when order_status='dropped' and spd_fraud_flag!=True and Est_Nav_bucket='c. Greater_than_500m' then order_id end) as Est_Nav_bucket_grt_500m
    
    
    
    from(
            select *,case when d2 is not NULL and fe_vs_nav>=-0.5 and fe_vs_nav<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and fe_vs_nav<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and fe_vs_nav>0.5 then 'c. Greater_than_500m' end as fe_vs_nav_bucket,
            case when  d2 is not NULL and nav_actual>=-0.5 and nav_actual<=0.5 then 'a. Within_500_m'
            when d2 is not NULL and nav_actual<-0.5 then 'b. Less_than_500m'
            when d2 is not NULL and nav_actual>0.5 then 'c. Greater_than_500m' end as nav_actual_bucket,
            case when Est_Nav>=-0.5 and Est_Nav<=0.5 then 'a. Within_500_m'
            when Est_Nav<-0.5 then 'b. Less_than_500m'
            when Est_Nav>0.5 then 'c. Greater_than_500m' end as Est_Nav_bucket
            from merged_tbl
            group by 1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16
    )
    group by 1,2,3,4
    """
    d1_df=pd.read_sql(all_dist,connection4)
    return d1_df

In [34]:
total_df_temp=pd.DataFrame()
for i in ['Link','Auto','Bike Lite']:
    print(i)
    d1=all_services_temporal_dist(start_date,end_date,city_name,i)
    total_df_temp=pd.concat([total_df_temp,d1])

Link
Auto
Bike Lite


In [26]:
total_df_temp[(total_df_temp['service_name']=='Link')&(total_df_temp['group_tc']=='TEST')]

,service_name,group_tc,lm_bucket,time_bucket,gross_orders,net_orders,route_diff_orders,fe_nav_bucket_within_500m,fe_nav_bucket_less_500m,fe_nav_bucket_grt_500m,nav_actual_bucket_within_500m,nav_actual_bucket_less_500m,nav_actual_bucket_grt_500m,Est_Nav_bucket_within_500m,Est_Nav_bucket_less_500m,Est_Nav_bucket_grt_500m
1,Link,TEST,Short,Rest_Morning,5058,2025,565,1686,70,193,1430,147,380,1454,100,471
2,Link,TEST,Long,Rest_Evening,1237,487,218,364,47,45,262,105,89,263,116,103
3,Link,TEST,Long,Morning_Peak,6816,4173,1844,3136,342,502,2310,879,778,2313,898,961
6,Link,TEST,Short,Morning_Peak,16328,8713,1929,7352,298,575,6380,506,1341,6762,398,1551
7,Link,TEST,Long,Evening_Peak,8805,4670,2193,3442,485,525,2438,1090,923,2463,1155,1052
8,Link,TEST,Medium,Afternoon,10105,6819,2273,5452,469,607,4365,935,1230,4530,918,1370
9,Link,TEST,Medium,Rest_Evening,1572,765,286,583,61,101,461,115,157,471,108,174
11,Link,TEST,Short,Rest_Evening,2202,1036,268,850,48,93,723,81,175,756,62,201
15,Link,TEST,Long,Afternoon,8372,5230,2337,3914,468,594,2836,1119,1007,2859,1203,1167
16,Link,TEST,Long,Rest_Morning,3921,1394,638,1046,118,158,750,253,313,742,270,382


In [35]:
total_df_temp.to_csv('raw_orders_v1.csv')